# Đồ án cuối khóa: Lời giải gợi ý

Dưới đây là một lời giải gợi ý cho đồ án cuối khóa. Lưu ý rằng đây chỉ là một trong nhiều cách tiếp cận. Miễn là thiết kế của bạn tuân thủ các nguyên tắc OOP, linh hoạt và giải quyết được bài toán, nó vẫn được coi là một lời giải tốt.

## 1. Import các thư viện

In [ ]:
import pandas as pd
import numpy as np
from abc import ABC, abstractmethod
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from typing import List, Dict, Any

## 2. Thiết kế các Class lõi

### BaseModel và SklearnModelWrapper
Chúng ta định nghĩa một "hợp đồng" cho tất cả các mô hình và một class cụ thể để bọc các mô hình của scikit-learn.

In [ ]:
class BaseModel(ABC):
    """Interface cho tất cả các model wrappers."""
    @abstractmethod
    def train(self, X_train: np.ndarray, y_train: np.ndarray) -> None:
        pass

    @abstractmethod
    def predict(self, X: np.ndarray) -> np.ndarray:
        pass

class SklearnModelWrapper(BaseModel):
    """Một wrapper cho các mô hình từ thư viện scikit-learn."""
    def __init__(self, model):
        self._model = model

    def train(self, X_train: np.ndarray, y_train: np.ndarray) -> None:
        print(f"Training {self._model.__class__.__name__}...")
        self._model.fit(X_train, y_train)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return self._model.predict(X)

### ModelFactory
Sử dụng Factory Pattern để tạo các mô hình một cách linh hoạt.

In [ ]:
class ModelFactory:
    """Tạo các model wrapper dựa trên tên và tham số."""
    def create_model(self, model_name: str, **params: Any) -> SklearnModelWrapper:
        model_map = {
            'logistic_regression': LogisticRegression,
            'decision_tree': DecisionTreeClassifier,
            'random_forest': RandomForestClassifier
        }
        
        model_class = model_map.get(model_name)
        if not model_class:
            raise ValueError(f"Model '{model_name}' is not supported.")
            
        # Thêm `random_state` nếu model hỗ trợ để đảm bảo kết quả có thể tái lập
        import inspect
        sig = inspect.signature(model_class)
        if 'random_state' in sig.parameters:
            params['random_state'] = 42

        model_instance = model_class(**params)
        return SklearnModelWrapper(model_instance)

## 3. Thiết kế các Class xử lý dữ liệu

In [ ]:
class DataLoader:
    """Chịu trách nhiệm tải dữ liệu từ một nguồn."""
    def load(self, url: str, column_names: List[str]) -> pd.DataFrame:
        print(f"Loading data from {url}...")
        df = pd.read_csv(url, header=None, names=column_names)
        return df

class DataPreprocessor:
    """Chịu trách nhiệm tiền xử lý dữ liệu."""
    def __init__(self, test_size: float = 0.2, random_state: int = 42):
        self._test_size = test_size
        self._random_state = random_state
        self._imputer = SimpleImputer(missing_values=0, strategy='mean')
        self._scaler = StandardScaler()

    def process(self, df: pd.DataFrame, target_column: str) -> tuple:
        print("Processing data...")
        X = df.drop(target_column, axis=1)
        y = df[target_column]

        # Xử lý giá trị 0 không hợp lệ
        cols_with_invalid_zeros = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
        X[cols_with_invalid_zeros] = self._imputer.fit_transform(X[cols_with_invalid_zeros])

        # Chia train/test
        X_train, X_test, y_train, y_test = train_test_split(
            X, y, test_size=self._test_size, random_state=self._random_state, stratify=y
        )

        # Scale dữ liệu
        X_train = self._scaler.fit_transform(X_train)
        X_test = self._scaler.transform(X_test)

        return X_train, X_test, y_train, y_test

## 4. Thiết kế Class đánh giá

In [ ]:
class Evaluator:
    """Đánh giá mô hình dựa trên các metrics được yêu cầu."""
    def evaluate(self, y_true: np.ndarray, y_pred: np.ndarray, metrics: List[str]) -> Dict[str, float]:
        print("Evaluating model...")
        results = {}
        metric_map = {
            'accuracy': accuracy_score,
            'precision': precision_score,
            'recall': recall_score,
            'f1': f1_score
        }
        for metric in metrics:
            metric_func = metric_map.get(metric)
            if metric_func:
                results[metric] = metric_func(y_true, y_pred)
        return results

## 5. Thiết kế Class "Nhạc trưởng" ExperimentRunner

In [ ]:
class ExperimentRunner:
    """Quản lý và thực thi toàn bộ quy trình thử nghiệm."""
    def __init__(self, config: Dict[str, Any]):
        self._config = config
        self._data_loader = DataLoader()
        self._preprocessor = DataPreprocessor()
        self._model_factory = ModelFactory()
        self._evaluator = Evaluator()
        self._results = []

    def run(self) -> None:
        print("===== Starting Experiment =====")
        # 1. Load data
        df = self._data_loader.load(self._config['data_url'], self._config['column_names'])

        # 2. Preprocess data
        X_train, X_test, y_train, y_test = self._preprocessor.process(df, 'Outcome')

        # 3. Loop through models, train, and evaluate
        for model_name, params in self._config['models_to_run'].items():
            print(f"\n--- Running model: {model_name} ---")
            # Create model
            model = self._model_factory.create_model(model_name, **params)
            
            # Train model
            model.train(X_train, y_train)
            
            # Predict
            predictions = model.predict(X_test)
            
            # Evaluate
            metrics = self._evaluator.evaluate(y_test, predictions, self._config['evaluation_metrics'])
            
            # Store results
            self._results.append({'model': model_name, **metrics})
        
        self._print_summary()
        print("\n===== Experiment Finished =====")

    def _print_summary(self) -> None:
        print("\n===== Experiment Summary =====")
        summary_df = pd.DataFrame(self._results)
        print(summary_df.to_string(index=False))

## 6. Cấu hình và Chạy thử nghiệm

In [ ]:
# Định nghĩa cấu hình cho thử nghiệm
experiment_config = {
    'data_url': 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv',
    'column_names': ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome'],
    'models_to_run': {
        'logistic_regression': {'C': 1.0, 'solver': 'liblinear'},
        'decision_tree': {'max_depth': 5},
        'random_forest': {'n_estimators': 100, 'max_depth': 5}
    },
    'evaluation_metrics': ['accuracy', 'precision', 'recall', 'f1']
}

# Khởi tạo và chạy runner
runner = ExperimentRunner(config=experiment_config)
runner.run()

## Phân tích lời giải

-   **Single Responsibility:** Mỗi class có một nhiệm vụ duy nhất và rõ ràng. `DataLoader` chỉ tải, `Evaluator` chỉ đo lường, `ExperimentRunner` chỉ điều phối.
-   **Open/Closed Principle:** Framework này "mở" cho việc mở rộng nhưng "đóng" cho việc sửa đổi. Để thêm một mô hình mới (ví dụ `XGBoost`), bạn chỉ cần cập nhật `ModelFactory` và `config`. Bạn không cần phải động đến `ExperimentRunner`.
-   **Dependency Inversion:** `ExperimentRunner` không phụ thuộc vào các mô hình cụ thể như `LogisticRegression`, mà phụ thuộc vào một abstraction là `BaseModel`. Điều này làm cho hệ thống rất linh hoạt.
-   **Flexibility:** Toàn bộ thử nghiệm được điều khiển bởi một dictionary `config`. Bạn có thể dễ dàng thay đổi bộ dữ liệu, danh sách mô hình, tham số, và các metrics chỉ bằng cách sửa đổi `config` mà không cần chạm vào code logic của các class.